# 🚀 ETF Portfolio Backtesting System - Full Production

**Run ทั้งหมดใน Notebook นี้ - ไม่ต้อง import external files!**

## 🎯 วิธีใช้งาน:
1. **Run Step 1:** ตั้งค่า Database
2. **Run Step 2:** Load ทุก functions
3. **Run Step 3:** เริ่มใช้งาน!

---

## ⚠️ Requirements:
- MySQL เปิดอยู่
- Database `etf_backtesting` พร้อมข้อมูล
- Python packages: `pip install pandas mysql-connector-python matplotlib ipywidgets`

---

# Step 1: Database Configuration

⚠️ **แก้ password ตรงนี้ให้ตรงกับ MySQL ของคุณ**

In [ ]:
# Database Configuration
DB_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456',  # ⚠️ แก้ตรงนี้!
    'database': 'etf_backtesting'
}

print("✅ Database configuration loaded")
print(f"   Host: {DB_CONFIG['host']}:{DB_CONFIG['port']}")
print(f"   User: {DB_CONFIG['user']}")
print(f"   Database: {DB_CONFIG['database']}")

# Test connection
import mysql.connector
try:
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()
    cursor.execute("SELECT VERSION()")
    version = cursor.fetchone()[0]
    cursor.close()
    conn.close()
    print(f"\n✅ MySQL Connected! (Version: {version})")
except Exception as e:
    print(f"\n❌ Connection Failed: {e}")
    print("\n💡 Please:")
    print("   1. Check MySQL is running")
    print("   2. Verify password above")
    print("   3. Ensure database exists")

# Step 2: Load All Functions

**ทุก functions ที่จำเป็นอยู่ใน cell นี้แล้ว!**

In [ ]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ===================================================================
# DATABASE UTILITIES
# ===================================================================

def get_connection():
    """สร้าง database connection"""
    try:
        return mysql.connector.connect(**DB_CONFIG)
    except Exception as e:
        print(f"❌ Connection error: {e}")
        return None

# ===================================================================
# PORTFOLIO FUNCTIONS
# ===================================================================

def show_portfolios():
    """แสดง portfolios ทั้งหมด"""
    conn = get_connection()
    if not conn:
        return None
    
    query = """
    SELECT 
        p.portfolio_id,
        p.name,
        p.description,
        COUNT(pe.ticker) as num_etfs,
        p.created_at
    FROM portfolios p
    LEFT JOIN portfolio_etfs pe ON p.portfolio_id = pe.portfolio_id
    GROUP BY p.portfolio_id
    ORDER BY p.portfolio_id
    """
    
    df = pd.read_sql(query, conn)
    conn.close()
    
    print("\n📁 PORTFOLIOS")
    print("=" * 80)
    display(df)
    print("=" * 80)
    return df

def show_portfolio_details(portfolio_id):
    """แสดงรายละเอียด portfolio"""
    conn = get_connection()
    if not conn:
        return None
    
    # Portfolio info
    query1 = f"SELECT * FROM portfolios WHERE portfolio_id = {portfolio_id}"
    portfolio = pd.read_sql(query1, conn)
    
    # ETFs in portfolio
    query2 = f"""
    SELECT 
        pe.ticker,
        e.name,
        e.category,
        pe.weight,
        e.expense_ratio
    FROM portfolio_etfs pe
    JOIN etfs e ON pe.ticker = e.ticker
    WHERE pe.portfolio_id = {portfolio_id}
    ORDER BY pe.weight DESC
    """
    etfs = pd.read_sql(query2, conn)
    conn.close()
    
    if portfolio.empty:
        print(f"❌ Portfolio ID {portfolio_id} not found!")
        return None
    
    print("\n" + "=" * 80)
    print(f"📊 Portfolio: {portfolio['name'].values[0]}")
    print(f"Description: {portfolio['description'].values[0]}")
    print("=" * 80)
    
    if not etfs.empty:
        display(etfs)
        print(f"\nTotal Weight: {etfs['weight'].sum():.2f}%")
        print(f"Number of ETFs: {len(etfs)}")
    else:
        print("⚠️ No ETFs in this portfolio")
    
    print("=" * 80)
    return etfs

# ===================================================================
# ETF FUNCTIONS
# ===================================================================

def show_etfs(category=None):
    """แสดง ETFs ทั้งหมด"""
    conn = get_connection()
    if not conn:
        return None
    
    if category:
        query = f"SELECT * FROM etfs WHERE category = '{category}' ORDER BY ticker"
    else:
        query = "SELECT * FROM etfs ORDER BY ticker"
    
    df = pd.read_sql(query, conn)
    conn.close()
    
    print(f"\n📊 ETFs" + (f" - {category}" if category else ""))
    print("=" * 80)
    display(df)
    print("=" * 80)
    print(f"Total: {len(df)} ETFs")
    print("=" * 80)
    return df

def show_etf_categories():
    """แสดง categories ทั้งหมด"""
    conn = get_connection()
    if not conn:
        return None
    
    query = """
    SELECT category, COUNT(*) as count
    FROM etfs
    GROUP BY category
    ORDER BY count DESC
    """
    df = pd.read_sql(query, conn)
    conn.close()
    
    print("\n📂 ETF Categories")
    print("=" * 80)
    display(df)
    print("=" * 80)
    return df

# ===================================================================
# PRICE DATA FUNCTIONS
# ===================================================================

def show_price_data(ticker, limit=10):
    """แสดงข้อมูลราคา"""
    conn = get_connection()
    if not conn:
        return None
    
    query = f"""
    SELECT date, open, high, low, close, volume, adjusted_close
    FROM daily_prices
    WHERE ticker = '{ticker}'
    ORDER BY date DESC
    LIMIT {limit}
    """
    df = pd.read_sql(query, conn)
    conn.close()
    
    if df.empty:
        print(f"❌ No data found for {ticker}")
        return None
    
    print(f"\n💹 Price Data: {ticker} (Latest {limit} days)")
    print("=" * 80)
    display(df)
    print("=" * 80)
    return df

def get_price_statistics(ticker, start_date=None, end_date=None):
    """คำนวณสถิติราคา"""
    conn = get_connection()
    if not conn:
        return None
    
    where_clause = f"WHERE ticker = '{ticker}'"
    if start_date:
        where_clause += f" AND date >= '{start_date}'"
    if end_date:
        where_clause += f" AND date <= '{end_date}'"
    
    query = f"""
    SELECT 
        ticker,
        COUNT(*) as trading_days,
        MIN(close) as min_price,
        MAX(close) as max_price,
        AVG(close) as avg_price,
        STDDEV(close) as std_price,
        MIN(date) as start_date,
        MAX(date) as end_date
    FROM daily_prices
    {where_clause}
    GROUP BY ticker
    """
    df = pd.read_sql(query, conn)
    conn.close()
    
    if df.empty:
        print(f"❌ No data found for {ticker}")
        return None
    
    print(f"\n📊 Price Statistics: {ticker}")
    print("=" * 80)
    for col in df.columns:
        print(f"{col:20s}: {df[col].values[0]}")
    print("=" * 80)
    return df

def plot_price_chart(ticker, start_date=None, end_date=None):
    """สร้าง price chart"""
    conn = get_connection()
    if not conn:
        return None
    
    where_clause = f"WHERE ticker = '{ticker}'"
    if start_date:
        where_clause += f" AND date >= '{start_date}'"
    if end_date:
        where_clause += f" AND date <= '{end_date}'"
    
    query = f"""
    SELECT date, close
    FROM daily_prices
    {where_clause}
    ORDER BY date
    """
    df = pd.read_sql(query, conn)
    conn.close()
    
    if df.empty:
        print(f"❌ No data found for {ticker}")
        return None
    
    plt.figure(figsize=(14, 6))
    plt.plot(df['date'], df['close'], linewidth=2, color='#2E86AB')
    plt.title(f'{ticker} Price Chart', fontsize=16, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Price ($)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Data points: {len(df):,}")
    print(f"💰 Price range: ${df['close'].min():.2f} - ${df['close'].max():.2f}")
    print(f"📈 Latest price: ${df['close'].iloc[-1]:.2f}")
    
    return df

# ===================================================================
# BACKTEST FUNCTIONS
# ===================================================================

def show_backtest_history(limit=10):
    """แสดง backtest history"""
    conn = get_connection()
    if not conn:
        return None
    
    query = f"""
    SELECT 
        b.backtest_id,
        p.name as portfolio_name,
        b.strategy_type,
        b.start_date,
        b.end_date,
        b.initial_capital,
        b.final_value,
        b.total_return,
        b.created_at
    FROM backtests b
    JOIN portfolios p ON b.portfolio_id = p.portfolio_id
    ORDER BY b.created_at DESC
    LIMIT {limit}
    """
    df = pd.read_sql(query, conn)
    conn.close()
    
    if df.empty:
        print("⚠️ No backtests found!")
        return None
    
    print(f"\n📈 Backtest History (Latest {limit})")
    print("=" * 80)
    display(df)
    print("=" * 80)
    return df

# ===================================================================
# STATISTICS FUNCTIONS
# ===================================================================

def show_system_statistics():
    """แสดงสถิติระบบ"""
    conn = get_connection()
    if not conn:
        return None
    
    cursor = conn.cursor()
    tables = ['etfs', 'daily_prices', 'portfolios', 'portfolio_etfs', 'backtests']
    stats = {}
    
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        stats[table] = cursor.fetchone()[0]
    
    cursor.close()
    conn.close()
    
    print("\n📊 SYSTEM STATISTICS")
    print("=" * 80)
    print(f"{'ETFs':30s}: {stats['etfs']:,}")
    print(f"{'Daily Price Records':30s}: {stats['daily_prices']:,}")
    print(f"{'Portfolios':30s}: {stats['portfolios']:,}")
    print(f"{'Portfolio Allocations':30s}: {stats['portfolio_etfs']:,}")
    print(f"{'Backtests':30s}: {stats['backtests']:,}")
    print("=" * 80)
    
    return stats

# ===================================================================
# COMPARISON FUNCTIONS
# ===================================================================

def compare_portfolios(portfolio_ids):
    """เปรียบเทียบ portfolios"""
    conn = get_connection()
    if not conn:
        return None
    
    ids_str = ','.join(map(str, portfolio_ids))
    
    query = f"""
    SELECT 
        p.portfolio_id,
        p.name,
        COUNT(pe.ticker) as num_etfs,
        AVG(e.expense_ratio) as avg_expense_ratio
    FROM portfolios p
    LEFT JOIN portfolio_etfs pe ON p.portfolio_id = pe.portfolio_id
    LEFT JOIN etfs e ON pe.ticker = e.ticker
    WHERE p.portfolio_id IN ({ids_str})
    GROUP BY p.portfolio_id
    """
    df = pd.read_sql(query, conn)
    conn.close()
    
    print("\n🔍 Portfolio Comparison")
    print("=" * 80)
    display(df)
    print("=" * 80)
    return df

def compare_etf_prices(tickers, start_date='2024-01-01'):
    """เปรียบเทียบราคา ETFs"""
    conn = get_connection()
    if not conn:
        return None
    
    tickers_str = "','" .join(tickers)
    
    query = f"""
    SELECT 
        ticker,
        AVG(close) as avg_price,
        MIN(close) as min_price,
        MAX(close) as max_price,
        STDDEV(close) as volatility,
        COUNT(*) as trading_days
    FROM daily_prices
    WHERE ticker IN ('{tickers_str}')
      AND date >= '{start_date}'
    GROUP BY ticker
    ORDER BY avg_price DESC
    """
    df = pd.read_sql(query, conn)
    conn.close()
    
    print(f"\n📊 ETF Price Comparison (from {start_date})")
    print("=" * 80)
    display(df)
    print("=" * 80)
    return df

print("\n" + "="*80)
print("✅ ALL FUNCTIONS LOADED SUCCESSFULLY!")
print("="*80)
print("\nAvailable functions:")
print("  📁 Portfolio: show_portfolios(), show_portfolio_details(id), compare_portfolios([ids])")
print("  📊 ETFs: show_etfs(), show_etf_categories()")
print("  💹 Prices: show_price_data(ticker), get_price_statistics(ticker), plot_price_chart(ticker)")
print("  📈 Backtests: show_backtest_history()")
print("  📊 Stats: show_system_statistics()")
print("  🔍 Compare: compare_portfolios([ids]), compare_etf_prices([tickers])")
print("="*80)

---

# 🎉 พร้อมใช้งานแล้ว!

ตอนนี้ใช้ functions ได้เลย - Run cells ด้านล่างตามที่ต้องการ

---

# 📊 Quick Actions - ใช้งานด่วน

In [ ]:
# แสดงสถิติระบบ
show_system_statistics()

In [ ]:
# ดู Portfolios ทั้งหมด
portfolios = show_portfolios()

In [ ]:
# ดูรายละเอียด Portfolio ID 1
portfolio_details = show_portfolio_details(1)

In [ ]:
# ดู ETFs ทั้งหมด
etfs = show_etfs()

In [ ]:
# ดู ETF Categories
categories = show_etf_categories()

# 💹 Price Data Analysis

In [ ]:
# ดูราคา SPY ล่าสุด 20 วัน
spy_prices = show_price_data('SPY', limit=20)

In [ ]:
# สถิติราคา SPY ปี 2024
spy_stats = get_price_statistics('SPY', start_date='2024-01-01')

In [ ]:
# สร้าง Price Chart - SPY ปี 2024
spy_chart = plot_price_chart('SPY', start_date='2024-01-01')

# 🔍 Comparison & Analysis

In [ ]:
# เปรียบเทียบ Portfolios 1, 2, 3
comparison = compare_portfolios([1, 2, 3])

In [ ]:
# เปรียบเทียบราคา ETFs หลักๆ ปี 2024
price_comparison = compare_etf_prices(['SPY', 'QQQ', 'VOO', 'VTI'], start_date='2024-01-01')

# 📈 Backtest History

In [ ]:
# ดู Backtest History 20 รายการล่าสุด
backtests = show_backtest_history(limit=20)

# 🎨 Custom Analysis Examples

In [ ]:
# กรอง ETFs ตาม Category
us_equity = show_etfs(category='US Equity')

In [ ]:
# สร้าง Price Chart หลายๆ ETF
tickers = ['SPY', 'QQQ', 'VOO']
for ticker in tickers:
    print(f"\n{'='*80}")
    plot_price_chart(ticker, start_date='2024-01-01')

In [ ]:
# Custom Query - ราคาเฉลี่ยแต่ละเดือนของ SPY ปี 2024
conn = get_connection()
query = """
SELECT 
    DATE_FORMAT(date, '%Y-%m') as month,
    AVG(close) as avg_price,
    MIN(close) as min_price,
    MAX(close) as max_price
FROM daily_prices
WHERE ticker = 'SPY'
  AND date >= '2024-01-01'
GROUP BY DATE_FORMAT(date, '%Y-%m')
ORDER BY month
"""
monthly_avg = pd.read_sql(query, conn)
conn.close()

print("\n📊 SPY Monthly Average Prices (2024)")
print("=" * 80)
display(monthly_avg)

# Plot
plt.figure(figsize=(12, 6))
plt.plot(monthly_avg['month'], monthly_avg['avg_price'], marker='o', linewidth=2)
plt.fill_between(monthly_avg['month'], monthly_avg['min_price'], monthly_avg['max_price'], alpha=0.3)
plt.title('SPY Monthly Price Range (2024)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Price ($)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

# 🎯 Interactive Dashboard (Optional)

Run cell นี้เพื่อใช้ Interactive Widgets!

---

In [ ]:
# สร้าง Interactive Dashboard
from ipywidgets import interact, Dropdown, IntSlider, DatePicker

# Get ticker list
conn = get_connection()
tickers_df = pd.read_sql("SELECT DISTINCT ticker FROM etfs ORDER BY ticker", conn)
conn.close()
ticker_list = tickers_df['ticker'].tolist()

def interactive_price_chart(ticker, days):
    """Interactive price chart"""
    show_price_data(ticker, limit=days)
    
    # Chart
    conn = get_connection()
    query = f"""
    SELECT date, close
    FROM daily_prices
    WHERE ticker = '{ticker}'
    ORDER BY date DESC
    LIMIT {days}
    """
    df = pd.read_sql(query, conn)
    conn.close()
    
    if not df.empty:
        df = df.sort_values('date')
        plt.figure(figsize=(12, 5))
        plt.plot(df['date'], df['close'], linewidth=2, color='#2E86AB')
        plt.title(f'{ticker} - Last {days} Days', fontsize=14, fontweight='bold')
        plt.xlabel('Date')
        plt.ylabel('Price ($)')
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

# Interactive widget
interact(interactive_price_chart,
         ticker=Dropdown(options=ticker_list, value='SPY', description='Ticker:'),
         days=IntSlider(min=10, max=365, step=10, value=30, description='Days:'))

---

# 📚 Summary

## ✅ สิ่งที่ทำได้ใน Notebook นี้:

1. **Portfolio Management**
   - ดู portfolios ทั้งหมด
   - ดูรายละเอียด portfolio
   - เปรียบเทียบ portfolios

2. **ETF Information**
   - ดู ETFs ทั้งหมด
   - กรองตาม category
   - ดู categories

3. **Price Data Analysis**
   - ดูข้อมูลราคาล่าสุด
   - คำนวณสถิติ
   - สร้าง charts
   - เปรียบเทียบราคา

4. **Backtest History**
   - ดู backtest results
   - เปรียบเทียบ strategies

5. **Custom Analysis**
   - SQL queries
   - Pandas analysis
   - Matplotlib visualizations

6. **Interactive Dashboard**
   - ipywidgets
   - Real-time updates

---

## 🚀 Next Steps:

- สร้าง custom analysis ของคุณเอง
- เพิ่ม charts และ visualizations
- ทดลอง compare portfolios
- วิเคราะห์ price trends

---

## 💡 Tips:

- ใช้ `shift + enter` เพื่อ run cell
- ใช้ `display(df)` แทน `print(df)` สำหรับ DataFrame
- สามารถแก้ไข functions ใน Step 2 ได้เลย
- ถ้าต้องการ reload functions ให้ run Step 2 ใหม่

---

**Happy Analyzing! 📊🚀**

---